# Arrhenius Kinetics of LLM Inference: 9-Step Empirical Protocol on MMLU-Pro

This notebook demonstrates the **Arrhenius Kinetics of LLM Inference** framework applied to MMLU-Pro overconfident-error instances.

**Core idea**: P(correct | temperature T) follows an Arrhenius-like law:
$$\log P(\text{correct}) = -E_a / T + \log A$$
where $E_a$ (activation energy) is a per-question difficulty parameter and $T$ is the sampling temperature.

**What this notebook does** (offline analysis on pre-collected data):
1. Load per-instance Arrhenius fit results from the pre-registered 9-step protocol
2. Re-run statistical analyses: R² distributions, Spearman correlations, T_thresh validation
3. Visualize the temperature–accuracy curves and key findings

**Original experiment**: 151 MMLU-Pro overconfident-error instances using `microsoft/phi-4` via OpenRouter, total cost $0.126. This demo loads the pre-computed results — no API calls needed.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is not pre-installed on Colab
_pip('loguru==0.7.3')

# Core scientific packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import json
import math
import os

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

## Data Loading

The demo data contains 19 curated examples from the full 700-example MMLU-Pro run:
- **16 OC (overconfident-error) instances** with full Arrhenius fit data
- **3 non-OC instances** showing normal behavior

Each OC instance has `predict_p_correct_by_T` (P(correct) at 7 temperatures), Arrhenius parameters (`Ea`, `R²`), and threshold estimates (`T_thresh`, `T_TURN`).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-ce2af9-arrhenius-kinetics-of-llm-inference-acti/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
examples = data["datasets"][0]["examples"]
metadata = data["metadata"]
print(f"Loaded {len(examples)} examples from {data['datasets'][0]['dataset']}")
print(f"Model: {metadata['model']}")
print(f"Verdict: {metadata['verdict']}  |  N_OC_instances: {metadata['n_oc_instances']}  |  Cost: ${metadata['cumulative_cost_usd']}")

## Configuration

All tunable parameters are defined here. The demo uses minimum values for fast execution.
The original full-run values are shown in comments.

In [ ]:
# Temperature grid used in the original experiment
TEMP_GRID = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
TURN_TEMPS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3]

# Bootstrap samples for R² confidence intervals
N_BOOT = 200          # original: 500

# Bootstrap samples for Spearman CIs
N_BOOT_SPEARMAN = 500  # original: 1000

# N values for T_thresh analysis (best-of-N)
N_VALUES = [4, 8, 16, 32]

# Minimum R² support points for valid Arrhenius fit
MIN_VALID_TEMPS = 3   # original: 3

## Parse OC Instances from Demo Data

Extract the overconfident-error instances that have Arrhenius fit data. Each instance has:
- `predict_p_correct_by_T`: JSON string with P(correct) at each temperature
- `predict_arrhenius_Ea`, `predict_arrhenius_R2`, `predict_arrhenius_log_A`: Arrhenius parameters
- `predict_T_pref`: temperature at which P(correct) was highest
- `predict_T_TURN`: entropy-inflection-based temperature estimate
- `predict_T_thresh_N16_simplified`: threshold temperature for best-of-16

In [ ]:
oc_instances = []
for ex in examples:
    if ex.get("predict_is_oc_error") != "true":
        continue
    # Parse the JSON-encoded p_correct_by_T string
    p_by_T = json.loads(ex["predict_p_correct_by_T"])
    p_by_T = {float(k): float(v) for k, v in p_by_T.items()}

    instance = {
        "qid": ex["metadata_question_id"],
        "subject": ex["metadata_subject"],
        "split": ex["metadata_split"],
        "Ea": float(ex["predict_arrhenius_Ea"]),
        "R2": float(ex["predict_arrhenius_R2"]),
        "log_A": float(ex["predict_arrhenius_log_A"]),
        "Delta": float(ex["predict_Delta"]),
        "T_pref": float(ex["predict_T_pref"]) if ex.get("predict_T_pref") else None,
        "T_TURN": float(ex["predict_T_TURN"]) if ex.get("predict_T_TURN") else None,
        "T_thresh_N16_simplified": float(ex["predict_T_thresh_N16_simplified"]) if ex.get("predict_T_thresh_N16_simplified") else None,
        "T_thresh_N16_approx": float(ex["predict_T_thresh_N16_approx"]) if ex.get("predict_T_thresh_N16_approx") else None,
        "T_thresh_N16_empirical_min": float(ex["predict_T_thresh_N16_empirical_min"]) if ex.get("predict_T_thresh_N16_empirical_min") else None,
        "R2_linear": float(ex.get("predict_R2_linear", 0) or 0),
        "R2_exp_T": float(ex.get("predict_R2_exp_T", 0) or 0),
        "R2_power_law": float(ex.get("predict_R2_power_law", 0) or 0),
        "p_by_T": p_by_T,
    }
    oc_instances.append(instance)

print(f"OC instances in demo: {len(oc_instances)}")
print(f"Subjects: {sorted(set(i['subject'] for i in oc_instances))}")
df = pd.DataFrame(oc_instances)
print(df[['qid', 'subject', 'Ea', 'R2', 'T_pref', 'T_thresh_N16_simplified']].to_string(index=False))

## Arrhenius Fitting Functions

These are the exact fitting functions from the original `method.py`. Given P(correct) at multiple temperatures,
they fit the Arrhenius model `log P = -Ea/T + log A` via OLS on `(1/T, log P)` pairs.

Alternative models (linear, exp_T, power law) are also fit for comparison.

In [ ]:
def fit_arrhenius(p_correct_by_T: dict):
    Ts = np.array(TEMP_GRID)
    Ps = np.array([p_correct_by_T.get(T, 0.0) for T in Ts])
    mask = Ps > 0
    if mask.sum() < MIN_VALID_TEMPS:
        return None
    inv_T = (1.0 / Ts[mask]).reshape(-1, 1)
    log_P = np.log(Ps[mask])
    reg = LinearRegression().fit(inv_T, log_P)
    log_P_pred = reg.predict(inv_T)
    ss_res = float(np.sum((log_P - log_P_pred) ** 2))
    ss_tot = float(np.sum((log_P - log_P.mean()) ** 2))
    R2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
    slope = float(reg.coef_[0])
    Ea = -slope
    log_A = float(reg.intercept_)
    return {
        "Ea": Ea,
        "log_A": log_A,
        "R2": R2,
        "n_valid": int(mask.sum()),
        "slope": slope,
        "valid_temps": Ts[mask].tolist(),
    }


def fit_alternatives(p_correct_by_T: dict) -> dict:
    Ts = np.array(TEMP_GRID)
    Ps = np.array([p_correct_by_T.get(T, 0.0) for T in Ts])
    mask = Ps > 0
    if mask.sum() < MIN_VALID_TEMPS:
        return {"linear": None, "exp_T": None, "power_law": None}
    Ts_v = Ts[mask]
    Ps_v = Ps[mask]

    def ols_r2(X: np.ndarray, y: np.ndarray) -> float:
        if len(np.unique(y)) < 2:
            return 0.0
        reg = LinearRegression().fit(X.reshape(-1, 1), y)
        pred = reg.predict(X.reshape(-1, 1))
        ss_res = float(np.sum((y - pred) ** 2))
        ss_tot = float(np.sum((y - y.mean()) ** 2))
        return 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

    r2_linear = ols_r2(Ts_v, Ps_v)
    r2_exp_T = ols_r2(Ts_v, np.log(Ps_v))
    log_Tv = np.log(Ts_v)
    r2_power = ols_r2(log_Tv, np.log(Ps_v)) if len(np.unique(log_Tv)) >= 2 else None
    return {"linear": r2_linear, "exp_T": r2_exp_T, "power_law": r2_power}


def bootstrap_R2(p_correct_by_T: dict, n_boot: int = N_BOOT) -> tuple:
    Ts = np.array(TEMP_GRID)
    Ps = np.array([p_correct_by_T.get(T, 0.0) for T in Ts])
    mask = Ps > 0
    if mask.sum() < MIN_VALID_TEMPS:
        return (None, None)
    inv_T = 1.0 / Ts[mask]
    log_P = np.log(Ps[mask])
    n = int(mask.sum())
    rng = np.random.default_rng(42)
    r2_boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        x_b = inv_T[idx].reshape(-1, 1)
        y_b = log_P[idx]
        if len(np.unique(y_b)) < 2:
            continue
        reg = LinearRegression().fit(x_b, y_b)
        pred = reg.predict(x_b)
        ss_res = float(np.sum((y_b - pred) ** 2))
        ss_tot = float(np.sum((y_b - y_b.mean()) ** 2))
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
        r2_boots.append(r2)
    if not r2_boots:
        return (None, None)
    arr = np.array(r2_boots)
    return (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))


def spearman_with_ci(x, y, n_boot: int = N_BOOT_SPEARMAN, seed: int = 42) -> dict:
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < 5:
        return {"rho": None, "p": None, "ci_low": None, "ci_high": None, "n": int(len(x))}
    rho, p_val = stats.spearmanr(x, y)
    rng = np.random.default_rng(seed)
    boots = []
    n = len(x)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        r, _ = stats.spearmanr(x[idx], y[idx])
        if np.isfinite(r):
            boots.append(r)
    arr = np.array(boots) if boots else np.array([rho])
    return {
        "rho": float(rho),
        "p": float(p_val),
        "ci_low": float(np.percentile(arr, 2.5)),
        "ci_high": float(np.percentile(arr, 97.5)),
        "n": n,
    }

## Re-fit Arrhenius Model on Demo Instances

Re-run the Arrhenius fitting on the OC instances loaded from the demo data. This validates that
our computed R² values match those stored in the pre-computed results, and demonstrates the
fitting pipeline end-to-end.

In [ ]:
refit_results = []

for inst in oc_instances:
    p_by_T = inst["p_by_T"]
    arr = fit_arrhenius(p_by_T)
    if arr is None:
        continue
    alts = fit_alternatives(p_by_T)
    r2_ci = bootstrap_R2(p_by_T, n_boot=N_BOOT)

    refit_results.append({
        "qid": inst["qid"],
        "subject": inst["subject"],
        "Ea": arr["Ea"],
        "log_A": arr["log_A"],
        "R2": arr["R2"],
        "R2_ci_low": r2_ci[0],
        "R2_ci_high": r2_ci[1],
        "R2_linear": alts["linear"],
        "R2_exp_T": alts["exp_T"],
        "R2_power_law": alts["power_law"],
        "n_valid": arr["n_valid"],
        "Delta": inst["Delta"],
        "T_pref": inst["T_pref"],
        "T_thresh_simplified": inst["T_thresh_N16_simplified"],
        "T_TURN": inst["T_TURN"],
        "p_by_T": p_by_T,
    })

print(f"Re-fitted {len(refit_results)} instances (of {len(oc_instances)} OC in demo)")
r2_vals = [r["R2"] for r in refit_results]
print(f"Median R² = {np.median(r2_vals):.4f}  (original: 0.8477)")
print(f"Mean   R² = {np.mean(r2_vals):.4f}  (original: 0.7437)")

## Step 5: Spearman ρ(E_a, Δ)

Test whether the Arrhenius activation energy $E_a$ correlates with the *logit gap* $\Delta = \text{logit}_{\text{wrong}} - \text{logit}_{\text{correct}}$ measured at T=1.

**Criterion C2**: ρ(Ea, Δ) > 0.5. The original result was ρ=0.106 (C2 **not met**).

In [ ]:
Ea_list = [r["Ea"] for r in refit_results]
Delta_list = [r["Delta"] for r in refit_results]
Tpref_list = [r["T_pref"] for r in refit_results if r["T_pref"] is not None]
Ea_for_Tpref = [r["Ea"] for r in refit_results if r["T_pref"] is not None]

sp_Ea_Delta = spearman_with_ci(Ea_list, Delta_list, n_boot=N_BOOT_SPEARMAN)
print(f"Step 5 — ρ(Ea, Δ) = {sp_Ea_Delta['rho']:.4f}  "
      f"95% CI [{sp_Ea_Delta['ci_low']:.3f}, {sp_Ea_Delta['ci_high']:.3f}]  "
      f"p={sp_Ea_Delta['p']:.4f}")
print(f"  C2 (ρ > 0.5): {'PASS' if sp_Ea_Delta['rho'] and sp_Ea_Delta['rho'] > 0.5 else 'FAIL'}")

## Step 6: T_thresh Validation

**T_thresh (simplified)** = E_a / ln(N) is the minimum temperature needed so that best-of-N sampling
has expected ≥1 correct. We validate that it is a **lower bound** on the empirically measured
minimum effective temperature.

**Criterion C3**: fraction(T_thresh < T_empirical_min) > 0.60.  
**Criterion C4**: fraction(T_TURN > T_thresh) ≥ 0.70 (the T_TURN operating window contains T_thresh).

In [ ]:
# For each instance with T_thresh and T_TURN, check lower-bound property
lb_checks = []
window_checks = []

for inst in oc_instances:
    T_thresh = inst["T_thresh_N16_simplified"]
    T_emp = inst["T_thresh_N16_empirical_min"]
    T_TURN = inst["T_TURN"]

    if T_thresh is not None and T_emp is not None:
        lb_checks.append(T_thresh < T_emp)
    if T_thresh is not None and T_TURN is not None:
        window_checks.append(T_TURN > T_thresh)

frac_lb = sum(lb_checks) / len(lb_checks) if lb_checks else None
frac_window = sum(window_checks) / len(window_checks) if window_checks else None

print(f"Step 6 — T_thresh lower-bound fraction (N=16): {frac_lb:.3f}  (n={len(lb_checks)})")
print(f"  C3 (fraction > 0.60): {'PASS' if frac_lb and frac_lb > 0.60 else 'FAIL'}")
print(f"Step 6 — Window fraction (T_TURN > T_thresh): {frac_window:.3f}  (n={len(window_checks)})")
print(f"  C4 (fraction >= 0.70): {'PASS' if frac_window and frac_window >= 0.70 else 'FAIL'}")

## Step 8: E_a Predicts T_pref

The key practical claim: **activation energy predicts the per-question optimal sampling temperature**.
Higher $E_a$ questions need higher temperatures to unlock correct responses.

**Criterion C6**: ρ(Ea, T_pref) > 0.3. The original result was ρ=0.674 (C6 **met**).

In [ ]:
sp_Ea_Tpref = spearman_with_ci(Ea_for_Tpref, Tpref_list, n_boot=N_BOOT_SPEARMAN)
print(f"Step 8 — ρ(Ea, T_pref) = {sp_Ea_Tpref['rho']:.4f}  "
      f"95% CI [{sp_Ea_Tpref['ci_low']:.3f}, {sp_Ea_Tpref['ci_high']:.3f}]  "
      f"p={sp_Ea_Tpref['p']:.6f}")
print(f"  C6 (ρ > 0.3): {'PASS' if sp_Ea_Tpref['rho'] and sp_Ea_Tpref['rho'] > 0.3 else 'FAIL'}")

## Aggregate Verdict

Reproduce the aggregate verdict logic from the original script: CONFIRM if ≥4 of 7 criteria are met.

In [ ]:
# Pull aggregate stats from the pre-computed metadata
agg = metadata["aggregate"]
step7 = metadata["step7_accuracy_comparison"]
acc = step7["accuracy"]

t_op_accs = [v for k, v in acc.items() if k.startswith("T_operating_delta")]
best_t_op = max(t_op_accs) if t_op_accs else None
fixed_07 = acc.get("fixed_T07")

print("=" * 60)
print("CONFIRMATION CRITERIA")
print("=" * 60)
for criterion, passed in agg["confirm_flags"].items():
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {status}  {criterion}")
print("="*60)
print(f"  Criteria met: {agg['n_criteria_confirmed']}/7")
print(f"  VERDICT: {agg['verdict']}")
print("="*60)
print(f"\nStep 7 accuracy:")
for k, v in acc.items():
    if isinstance(v, float):
        print(f"  {k:<35} {v:.3f}")

## Visualization

Four panels:
1. **Temperature–accuracy curves** for top-5 instances by R² (Arrhenius fit overlaid)
2. **R² distribution** across all OC instances in demo
3. **Ea vs T_pref scatter** (Step 8 — C6 confirmed)
4. **T_thresh vs T_TURN** (Step 6 — window fraction)

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# ── Panel 1: Temperature-accuracy curves (top-5 by R²) ─────────────────────
ax1 = fig.add_subplot(gs[0, 0])
top5 = sorted(refit_results, key=lambda x: x["R2"], reverse=True)[:5]
colors = plt.cm.tab10.colors
T_fine = np.linspace(0.05, 1.1, 200)

for i, inst in enumerate(top5):
    # Observed points
    Ts = sorted(inst["p_by_T"].keys())
    Ps = [inst["p_by_T"][T] for T in Ts]
    ax1.plot(Ts, Ps, 'o', color=colors[i], markersize=6)
    # Arrhenius fit curve
    Ea, log_A = inst["Ea"], inst["log_A"]
    P_fit = [np.exp(-Ea / T + log_A) for T in T_fine]
    P_fit_clipped = np.clip(P_fit, 0, 1)
    label = f"Q{inst['qid']} ({inst['subject'][:6]}) R²={inst['R2']:.3f}"
    ax1.plot(T_fine, P_fit_clipped, '-', color=colors[i], alpha=0.7, label=label)

ax1.set_xlabel("Temperature T")
ax1.set_ylabel("P(correct)")
ax1.set_title("Arrhenius Fits: Top-5 by R² (demo subset)")
ax1.legend(fontsize=7, loc="upper left")
ax1.set_xlim(0, 1.15)
ax1.set_ylim(-0.02, 1.05)
ax1.grid(True, alpha=0.3)

# ── Panel 2: R² distribution ────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
r2_all = [r["R2"] for r in refit_results]
ax2.hist(r2_all, bins=10, color='steelblue', edgecolor='white', alpha=0.85)
ax2.axvline(np.median(r2_all), color='red', linestyle='--', linewidth=2,
            label=f"Median R²={np.median(r2_all):.3f}")
ax2.axvline(0.85, color='orange', linestyle=':', linewidth=2, label="C1 threshold (0.85)")
ax2.set_xlabel("R² (Arrhenius fit)")
ax2.set_ylabel("Count")
ax2.set_title("R² Distribution (demo: 16 OC instances)")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# ── Panel 3: Ea vs T_pref ───────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
Ea_vals = [r["Ea"] for r in refit_results if r["T_pref"] is not None]
Tp_vals = [r["T_pref"] for r in refit_results if r["T_pref"] is not None]

ax3.scatter(Ea_vals, Tp_vals, color='darkorchid', alpha=0.8, s=70, zorder=3)
if len(Ea_vals) >= 2:
    z = np.polyfit(Ea_vals, Tp_vals, 1)
    p_trend = np.poly1d(z)
    x_range = np.linspace(min(Ea_vals), max(Ea_vals), 100)
    ax3.plot(x_range, p_trend(x_range), 'r--', alpha=0.6, linewidth=1.5)
rho_label = f"ρ={sp_Ea_Tpref['rho']:.3f}" if sp_Ea_Tpref['rho'] else "ρ=N/A"
ax3.set_xlabel("Activation Energy Ea")
ax3.set_ylabel("T_pref (optimal temperature)")
ax3.set_title(f"Step 8: Ea predicts T_pref ({rho_label}, C6 confirmed)")
ax3.grid(True, alpha=0.3)

# ── Panel 4: T_thresh vs T_TURN scatter ─────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ts_vals = [inst["T_thresh_N16_simplified"] for inst in oc_instances
           if inst["T_thresh_N16_simplified"] is not None and inst["T_TURN"] is not None]
tt_vals = [inst["T_TURN"] for inst in oc_instances
           if inst["T_thresh_N16_simplified"] is not None and inst["T_TURN"] is not None]

# Color by T_TURN > T_thresh (window condition)
colors_pts = ['green' if tt > ts else 'red' for ts, tt in zip(ts_vals, tt_vals)]
ax4.scatter(ts_vals, tt_vals, c=colors_pts, alpha=0.8, s=70, zorder=3)
# Diagonal reference line
lim = max(max(ts_vals, default=1), max(tt_vals, default=1)) * 1.1
ax4.plot([0, lim], [0, lim], 'k--', alpha=0.4, linewidth=1, label="T_TURN = T_thresh")
n_above = sum(1 for tt, ts in zip(tt_vals, ts_vals) if tt > ts)
frac_above = n_above / len(tt_vals) if tt_vals else 0
ax4.set_xlabel("T_thresh (simplified, N=16)")
ax4.set_ylabel("T_TURN (entropy inflection)")
ax4.set_title(f"Step 6: Window fraction = {frac_above:.2f} (C4: ≥0.70)\n"
              f"Green = T_TURN > T_thresh ({n_above}/{len(tt_vals)})")
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

plt.suptitle(
    "Arrhenius Kinetics of LLM Inference: MMLU-Pro Demo\n"
    f"Model: {metadata['model']} | Verdict: {metadata['verdict']} (4/7 criteria)",
    fontsize=12, fontweight='bold', y=1.01
)
plt.savefig("arrhenius_demo_results.png", dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved to arrhenius_demo_results.png")

## Summary Table

In [ ]:
print("=" * 70)
print("ARRHENIUS KINETICS OF LLM INFERENCE — SUMMARY")
print("=" * 70)
print(f"  Model: {metadata['model']}")
print(f"  Dataset: TIGER-Lab/MMLU-Pro")
print(f"  OC instances (full run): {metadata['n_oc_instances']}")
print(f"  Total API cost: ${metadata['cumulative_cost_usd']}")
print()
print("  R² Distribution (instances with valid Arrhenius fit):")
print(f"    Median R² = {np.median(r2_all):.4f}   [C1 threshold: > 0.85]")
print(f"    Mean   R² = {np.mean(r2_all):.4f}")
print()
print(f"  Step 5: ρ(Ea, Δ)    = {sp_Ea_Delta['rho']:.4f}   [C2 threshold: > 0.5]")
print(f"  Step 6: T_thresh LB  = {frac_lb:.3f}    [C3 threshold: > 0.60]")
print(f"  Step 6: Window frac  = {frac_window:.3f}    [C4 threshold: >= 0.70]")
print(f"  Step 8: ρ(Ea, Tpref) = {sp_Ea_Tpref['rho']:.4f}   [C6 threshold: > 0.3]")
print()
print(f"  Step 7 Accuracy (from pre-computed data):")
print(f"    T_operating best  = {best_t_op:.3f}")
print(f"    Fixed T=0.7       = {fixed_07:.3f}")
print(f"    Gain              = +{(best_t_op - fixed_07)*100:.1f}pp   [C7: T_operating > fixed]")
print()
print(f"  VERDICT: {metadata['verdict']} ({agg['n_criteria_confirmed']}/7 criteria confirmed)")
print("=" * 70)